In [1]:
# ============================================================
# MultiModalCancerTherapy
# 03 - Clinical Model Training
# Step 1 - Load Clinical Dataset
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project Root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print("Project Root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# Clinical Dataset Path
# ------------------------------------------------------------

CLINICAL_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "clinical"
    / "Thyroid Disease Data"
    / "thyroidDF.csv"
)

print("\nClinical Dataset:")
print(CLINICAL_DATA_PATH)


# ------------------------------------------------------------
# Check Dataset
# ------------------------------------------------------------

if not CLINICAL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Clinical dataset not found:\n{CLINICAL_DATA_PATH}"
    )


# ------------------------------------------------------------
# Load Dataset
# ------------------------------------------------------------

clinical_df = pd.read_csv(
    CLINICAL_DATA_PATH
)


# ------------------------------------------------------------
# Basic Information
# ------------------------------------------------------------

print("\nClinical dataset loaded successfully.")

print("Rows:", clinical_df.shape[0])
print("Columns:", clinical_df.shape[1])

print("\nDataset columns:")
print(clinical_df.columns.tolist())

Project Root:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure

Clinical Dataset:
D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\datasets\raw\clinical\Thyroid Disease Data\thyroidDF.csv

Clinical dataset loaded successfully.
Rows: 9172
Columns: 31

Dataset columns:
['age', 'sex', 'on_thyroxine', 'query_on_thyroxine', 'on_antithyroid_meds', 'sick', 'pregnant', 'thyroid_surgery', 'I131_treatment', 'query_hypothyroid', 'query_hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH_measured', 'TSH', 'T3_measured', 'T3', 'TT4_measured', 'TT4', 'T4U_measured', 'T4U', 'FTI_measured', 'FTI', 'TBG_measured', 'TBG', 'referral_source', 'target', 'patient_id']


In [2]:
# ============================================================
# Step 2 - Clinical Data Preparation and Train/Test Split
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


# ------------------------------------------------------------
# 1. Keep only records with a recorded diagnosis
# ------------------------------------------------------------

diagnosis_df = clinical_df[
    clinical_df["target"] != "-"
].copy()

print("Original records:", len(clinical_df))
print("Records with diagnosis:", len(diagnosis_df))


# ------------------------------------------------------------
# 2. Remove singleton target classes
# ------------------------------------------------------------

target_counts = (
    diagnosis_df["target"]
    .value_counts()
)

valid_classes = target_counts[
    target_counts >= 2
].index

model_df = diagnosis_df[
    diagnosis_df["target"].isin(valid_classes)
].copy()

print(
    "Records used for model training:",
    len(model_df)
)

print(
    "Number of target classes:",
    model_df["target"].nunique()
)


# ------------------------------------------------------------
# 3. Separate features and target
# ------------------------------------------------------------

X = model_df.drop(
    columns=[
        "target",
        "patient_id",
        "TBG"
    ]
).copy()

y = model_df["target"].copy()


# ------------------------------------------------------------
# 4. Convert binary t/f columns to 1/0
# ------------------------------------------------------------

binary_columns = [
    "on_thyroxine",
    "query_on_thyroxine",
    "on_antithyroid_meds",
    "sick",
    "pregnant",
    "thyroid_surgery",
    "I131_treatment",
    "query_hypothyroid",
    "query_hyperthyroid",
    "lithium",
    "goitre",
    "tumor",
    "hypopituitary",
    "psych",
    "TSH_measured",
    "T3_measured",
    "TT4_measured",
    "T4U_measured",
    "FTI_measured",
    "TBG_measured"
]

for column in binary_columns:

    X[column] = X[column].map({
        "t": 1,
        "f": 0
    })


# ------------------------------------------------------------
# 5. Encode sex
# ------------------------------------------------------------

X["sex"] = X["sex"].map({
    "M": 1,
    "F": 0
})


# ------------------------------------------------------------
# 6. Encode target labels
# ------------------------------------------------------------

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print("\nTarget classes:")
print(
    list(target_encoder.classes_)
)


# ------------------------------------------------------------
# 7. Define feature groups
# ------------------------------------------------------------

numeric_features = [
    "age",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI"
]

categorical_features = [
    "sex",
    "referral_source"
]


# ------------------------------------------------------------
# 8. Create preprocessing pipeline
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

clinical_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
], remainder="passthrough")


# ------------------------------------------------------------
# 9. Train/Test split
# ------------------------------------------------------------

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain/Test split:")
print("Training:", X_train_raw.shape)
print("Testing :", X_test_raw.shape)


# ------------------------------------------------------------
# 10. Fit ONLY on training data
# ------------------------------------------------------------

X_train_processed = (
    clinical_preprocessor.fit_transform(
        X_train_raw
    )
)

X_test_processed = (
    clinical_preprocessor.transform(
        X_test_raw
    )
)


# ------------------------------------------------------------
# 11. Verification
# ------------------------------------------------------------

print("\nProcessed data:")
print(
    "Training:",
    X_train_processed.shape
)

print(
    "Testing :",
    X_test_processed.shape
)

print(
    "\nRemaining training missing values:",
    np.isnan(
        X_train_processed.toarray()
        if hasattr(X_train_processed, "toarray")
        else X_train_processed
    ).sum()
)

print(
    "Remaining testing missing values:",
    np.isnan(
        X_test_processed.toarray()
        if hasattr(X_test_processed, "toarray")
        else X_test_processed
    ).sum()
)

print("\nPreprocessing completed successfully.")

Original records: 9172
Records with diagnosis: 2401
Records used for model training: 2396
Number of target classes: 26

Target classes:
['A', 'AK', 'B', 'C', 'C|I', 'D', 'F', 'FK', 'G', 'GI', 'GK', 'H|K', 'I', 'J', 'K', 'KJ', 'L', 'M', 'MI', 'MK', 'N', 'O', 'P', 'Q', 'R', 'S']

Train/Test split:
Training: (1916, 28)
Testing : (480, 28)

Processed data:
Training: (1916, 34)
Testing : (480, 34)

Remaining training missing values: 0
Remaining testing missing values: 0

Preprocessing completed successfully.


In [3]:
# ============================================================
# Step 3 - Train Clinical Random Forest Model
# ============================================================

from sklearn.ensemble import RandomForestClassifier

clinical_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Starting Clinical Random Forest training...")

clinical_model.fit(
    X_train_processed,
    y_train
)

print("\nClinical Random Forest training completed successfully.")

print("\nModel parameters:")
print("Estimators:", clinical_model.n_estimators)
print("Class weight:", clinical_model.class_weight)

Starting Clinical Random Forest training...

Clinical Random Forest training completed successfully.

Model parameters:
Estimators: 300
Class weight: balanced
